# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore a Croissant-structured dataset using the [`mlcroissant`](https://croissantml.org) library.

### Dataset Source
FAIR² dataset: _Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya._

- Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and prepare to access records from each record set using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object (not a dictionary)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available *record sets*, *fields*, and their `@id$s` using the dataset schema. All references in this notebook will use entity `@id`s.

In [ ]:
# List record sets with @ids
print("\nAvailable Record Sets:")
record_sets = list(dataset.record_sets)

if not record_sets:
    print('No record sets were found in this dataset — please check the schema or distribution!')
else:
    for rs in record_sets:
        print(f"- @id: {rs.id}, name: {rs.name}")

# For demonstration, get fields and columns for the first record set
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nFields for record set @id {first_record_set.id}:")
    for field in first_record_set.fields:
        print(f"  - field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    print("\nColumns (data source mapping) for this record set:")
    for column in first_record_set.columns:
        print(f"  - column @id: {column.id}, name: {column.name}, source: {getattr(column, 'source', None)}")

## 3. Data Extraction
Load records for each record set into a DataFrame. Use each record set's `@id`, and assign each loaded DataFrame by `@id` for downstream analysis.

**Note:** If the record sets list is empty, this section will not load data; otherwise, each DataFrame will be referenced by its record set `@id`.

In [ ]:
dfs = {}

# Identify all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # Load records for each record set @id
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set: {record_set_id}")
    if not dfs[record_set_id].empty:
        print(f"Columns for {record_set_id}: {dfs[record_set_id].columns.tolist()}")

# For further analysis, pick the first non-empty record set
if dfs:
    for rsid in dfs:
        if not dfs[rsid].empty:
            main_record_set_id = rsid
            print(f"\nUsing record set {main_record_set_id} for downstream examples.")
            break
    else:
        main_record_set_id = None
        print('No populated record sets found.')
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Apply data processing, including filtering and normalization. All field/column references use their `@id`. Adjust numeric and grouping field choices as appropriate for your data.

In [ ]:
import numpy as np

# EDA proceeds only if data was loaded
if main_record_set_id and not dfs[main_record_set_id].empty:
    df = dfs[main_record_set_id]
    # Select a numeric field for filtering/normalizing (choose first available if unsure)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols)==0:
        print('No numeric columns detected in record set:', main_record_set_id)
    else:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].std()!=0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):\n")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Select a grouping field (use first categorical/text column, if available)
        possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
        else:
            print('No suitable grouping field found!')
else:
    print('No main dataframe for EDA.')

## 5. Visualization

Visualize data distributions or relationships between fields using record set and field `@id`s. The plots below use the columns loaded in earlier steps and depend on available numeric and categorical variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not dfs[main_record_set_id].empty:
    df = dfs[main_record_set_id]
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        # Plot histogram of first numeric field
        field_id = numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field_id].dropna(), bins=20, kde=True)
        plt.title(f'Distribution of {field_id} (@id)')
        plt.xlabel(field_id)
        plt.show()

    # If grouping field is available, show boxplot
    group_fields = [col for col in df.columns if df[col].dtype == object]
    if len(group_fields) > 0 and len(numeric_cols) > 0:
        group_field = group_fields[0]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=field_id, data=df)
        plt.title(f'{field_id} grouped by {group_field} (using @id)')
        plt.xlabel(group_field)
        plt.ylabel(field_id)
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print('No data loaded — cannot plot visualizations.')

## 6. Conclusion

In this notebook, we demonstrated how to load and inspect a Croissant-structured dataset using the `mlcroissant` library. All entities—including record sets, fields, and columns—were referenced using their `@id`s, ensuring reproducibility and alignment with FAIR data principles.

We previewed available record sets, loaded tabular records for further exploration, filtered and normalized numeric variables, and visualized key data distributions. This workflow forms a reproducible foundation for in-depth analyses or integration with machine learning workflows based on well-defined schema metadata.

*For more information, see [mlcroissant documentation](https://croissantml.org) or the [FAIR² data package](https://open.sen.science/doi/10.71728/senscience.y7m0-f273).*